# 2. BaTiO₃ single points: Ti displacement, polarization, and BECs

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/02_batio3_single_point.ipynb)

The notebook shifts Ti along z inside its oxygen cage and evaluates the bundled five-atom teaching checkpoint at several fields. It plots polarization and the Ti $Z^*_{zz}$ response.

The toy training set has energy, force, and polarization labels, but no BEC labels. The BEC values below are derivatives produced by the model, not supervised BEC predictions. A finite-difference comparison checks their derivative consistency.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACEField-Landau.model"
TRAIN = DATA_DIR / "batio3/macefield_batio3_toy_train.extxyz"
source = read(TRAIN, index=0)
source_displacement = float(source.info["REF_mode_displacement"])
ti_index = source.get_chemical_symbols().index("Ti")
cell = source.cell.copy()
volume = source.get_volume()
source.info.clear()
for key in list(source.arrays):
    if key not in {"numbers", "positions"}:
        source.arrays.pop(key)

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=device,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=True,
    compute_polarizability=False,
)

def structure_at(displacement):
    atoms = source.copy()
    atoms.positions[ti_index, 2] += float(displacement) - source_displacement
    atoms.info.clear()
    return atoms

displacements = np.linspace(-0.18, 0.18, 13)
fields = [-0.06, 0.0, 0.06]
polarizations = {field: [] for field in fields}
ti_zz = []

for displacement in displacements:
    atoms = structure_at(displacement)
    calc.electric_field = [0.0, 0.0, 0.0]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    becs = np.asarray(calc.results["becs"]).reshape(len(atoms), 3, 3)
    ti_zz.append(float(becs[ti_index, 2, 2]))

    for field_z in fields:
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        atoms.calc = calc
        atoms.get_potential_energy()
        polarizations[field_z].append(
            float(calc.results["polarization"][2])
        )

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for field_z in fields:
    axes[0].plot(displacements, polarizations[field_z], "o-", label=f"Ez={field_z:+.2f} V/Å")
axes[0].set(xlabel="Ti displacement along z (Å)", ylabel="Pz (e/Å²)",
            title="Field-dependent polarization")
axes[0].legend()
axes[1].plot(displacements, ti_zz, "o-", color="#9b4f96")
axes[1].set(xlabel="Ti displacement along z (Å)", ylabel="Ti Z*zz (e)",
            title="Model-derived Born effective charge")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Check the BEC against a finite difference

For a cell polarization density, the Born charge includes the cell volume: $Z^*_{zz}=\Omega\,\partial P_z/\partial r_{\mathrm{Ti},z}$. The comparison below is at zero field and one displacement.


In [ ]:
delta = 1.0e-3
d0 = 0.04
field_z = 0.0

atoms0 = structure_at(d0)
calc.electric_field = [0.0, 0.0, field_z]
calc.reset()
atoms0.calc = calc
atoms0.get_potential_energy()
z_model = float(
    np.asarray(calc.results["becs"]).reshape(len(atoms0), 3, 3)[ti_index, 2, 2]
)

p_values = []
for displacement in (d0 - delta, d0 + delta):
    atoms = structure_at(displacement)
    calc.electric_field = [0.0, 0.0, field_z]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    p_values.append(float(calc.results["polarization"][2]))

z_fd = volume * (p_values[1] - p_values[0]) / (2 * delta)
print("model Ti Z*zz (e):", z_model)
print("finite-difference Ti Z*zz (e):", z_fd)
